# LlamaIndex + CoordiNode: PropertyGraphIndex

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/structured-world/coordinode-python/blob/main/demo/notebooks/01_llama_index_property_graph.ipynb)

Demonstrates `CoordinodePropertyGraphStore` as a backend for LlamaIndex `PropertyGraphIndex`.

**What works right now:**
- `upsert_nodes` / `upsert_relations` — idempotent MERGE (safe to call multiple times)
- `get()` — look up nodes by ID or properties
- `get_triplets()` — all edges (wildcard) or filtered by relation type / entity name
- `get_rel_map()` — outgoing relations for a set of nodes (depth=1)
- `structured_query()` — arbitrary Cypher pass-through
- `delete()` — remove nodes by id or name
- `get_schema()` — live text schema of the graph

**Environments:**
- **Google Colab** — uses `coordinode-embedded` (in-process Rust engine, no server needed). First run compiles from source (~5 min); subsequent runs use the pip cache.
- **Local / Docker Compose** — connects to a running CoordiNode server via gRPC.

## Install dependencies

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

# Install coordinode-embedded in Colab only (requires Rust build).
if IN_COLAB:
    # Install Rust toolchain via rustup (https://rustup.rs).
    # Colab's apt packages ship rustc ≤1.75, which cannot build coordinode-embedded
    # (requires Rust ≥1.80 for maturin/pyo3). apt-get is not a viable alternative here.
    # Download the installer to a temp file and execute it explicitly — this avoids
    # piping remote content directly into a shell while maintaining HTTPS/TLS security
    # through Python's default ssl context (cert-verified, TLS 1.2+).
    # SHA256 pinning of rustup-init is intentionally omitted: rustup.rs does not
    # publish a stable per-release checksum for sh.rustup.rs itself (only for
    # platform-specific rustup-init binaries), and pinning a hash here would break
    # silently on every rustup release. The HTTPS/TLS verification + temp-file
    # execution (not piped to shell) is the rustup team's recommended trust model.
    # No additional env-var gate (e.g. COORDINODE_ENABLE_RUSTUP) is needed:
    # the `IN_COLAB` check above already ensures this block never runs outside
    # Colab sessions, so there is no risk of unintentional execution in local
    # or server environments.
    import ssl as _ssl, tempfile as _tmp, urllib.request as _ur

    _ctx = _ssl.create_default_context()
    with _tmp.NamedTemporaryFile(mode="wb", suffix=".sh", delete=False) as _f:
        with _ur.urlopen("https://sh.rustup.rs", context=_ctx, timeout=30) as _r:
            _f.write(_r.read())
        _rustup_path = _f.name
    try:
        subprocess.run(["/bin/sh", _rustup_path, "-s", "--", "-y", "-q"], check=True, timeout=300)
    finally:
        os.unlink(_rustup_path)
    # Add cargo to PATH so maturin/pip can find it.
    _cargo_bin = os.path.expanduser("~/.cargo/bin")
    os.environ["PATH"] = f"{_cargo_bin}{os.pathsep}{os.environ.get('PATH', '')}"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "maturin"], check=True, timeout=300)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/structured-world/coordinode-python.git@e3ff279#subdirectory=coordinode-embedded",
        ],
        check=True,
        timeout=600,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "coordinode",
        "llama-index-graph-stores-coordinode",
        "llama-index-core",
        "nest_asyncio",
    ],
    check=True,
    timeout=300,
)

import nest_asyncio

nest_asyncio.apply()

print("SDK installed")

## Adapter for embedded mode

`LocalClient` (embedded engine) has the same `.cypher()` API as `CoordinodeClient`.
The `_EmbeddedAdapter` below adds the extra methods that the graph store expects
when it receives a pre-built `client=` object.

In [ ]:
class _EmbeddedAdapter:
    """Thin wrapper around LocalClient that adds CoordinodeClient-compatible methods."""

    def __init__(self, local_client):
        self._lc = local_client

    def cypher(self, query, params=None):
        return self._lc.cypher(query, params or {})

    def get_schema_text(self):
        lbls = self._lc.cypher("MATCH (n) UNWIND labels(n) AS lbl RETURN DISTINCT lbl ORDER BY lbl")
        rels = self._lc.cypher("MATCH ()-[r]->() RETURN DISTINCT type(r) AS t ORDER BY t")
        lines = ["Node labels:"]
        for r in lbls:
            lines.append(f"  - {r['lbl']}")
        lines.append("\nEdge types:")
        for r in rels:
            lines.append(f"  - {r['t']}")
        return "\n".join(lines)

    # Vector search not available in embedded mode — requires running CoordiNode server.

    def close(self):
        self._lc.close()

    def get_labels(self):
        # Schema introspection not available in embedded mode.

        return []

    def get_edge_types(self):
        # Schema introspection not available in embedded mode.
        return []

## Connect to CoordiNode

In [ ]:
import os, socket


def _port_open(port):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True
    except OSError:
        return False


GRPC_PORT = int(os.environ.get("COORDINODE_PORT", "7080"))

if os.environ.get("COORDINODE_ADDR"):
    COORDINODE_ADDR = os.environ["COORDINODE_ADDR"]
    from coordinode import CoordinodeClient

    client = CoordinodeClient(COORDINODE_ADDR)
    print(f"Connected to {COORDINODE_ADDR}")
elif _port_open(GRPC_PORT):
    COORDINODE_ADDR = f"localhost:{GRPC_PORT}"
    from coordinode import CoordinodeClient

    client = CoordinodeClient(COORDINODE_ADDR)
    print(f"Connected to {COORDINODE_ADDR}")
else:
    # No server available — use the embedded in-process engine.
    # Works without Docker or any external service; data is in-memory.
    try:
        from coordinode_embedded import LocalClient
    except ImportError as exc:
        raise RuntimeError(
            "coordinode-embedded is not installed. "
            "Run: pip install git+https://github.com/structured-world/coordinode-python.git@e3ff279#subdirectory=coordinode-embedded"
            " — or start a CoordiNode server and set COORDINODE_ADDR."
        ) from exc

    _lc = LocalClient(":memory:")
    client = _EmbeddedAdapter(_lc)
    print("Using embedded LocalClient (in-process)")

## Create the property graph store

Pass the pre-built `client=` object so the store doesn't open a second connection.

In [ ]:
from llama_index.graph_stores.coordinode import CoordinodePropertyGraphStore
from llama_index.core.graph_stores.types import EntityNode, Relation

store = CoordinodePropertyGraphStore(client=client)
print("Connected. Schema:")
print(store.get_schema()[:300])

## 1. Upsert nodes and relations

Each node gets a unique tag so this notebook can run multiple times without conflicts.

In [ ]:
import uuid

tag = uuid.uuid4().hex[:6]

nodes = [
    EntityNode(label="Person", name=f"Alice-{tag}", properties={"role": "researcher", "field": "AI"}),
    EntityNode(label="Person", name=f"Bob-{tag}", properties={"role": "engineer", "field": "ML"}),
    EntityNode(label="Topic", name=f"GraphRAG-{tag}", properties={"domain": "knowledge graphs"}),
]
store.upsert_nodes(nodes)
print("Upserted nodes:", [n.name for n in nodes])

alice, bob, graphrag = nodes
relations = [
    Relation(label="RESEARCHES", source_id=alice.id, target_id=graphrag.id, properties={"since": 2023}),
    Relation(label="COLLABORATES", source_id=alice.id, target_id=bob.id),
    Relation(label="IMPLEMENTS", source_id=bob.id, target_id=graphrag.id),
]
store.upsert_relations(relations)
print("Upserted relations:", [r.label for r in relations])

## 2. get_triplets — all edges from a node (wildcard)

In [ ]:
triplets = store.get_triplets(entity_names=[f"Alice-{tag}"])
print(f"Triplets for Alice-{tag}:")
for src, rel, dst in triplets:
    print(f"  {src.name}  --[{rel.label}]-->  {dst.name}")

## 3. get_rel_map — relations for a set of nodes

In [ ]:
found_alice = store.get(properties={"name": f"Alice-{tag}"})
rel_map = store.get_rel_map(found_alice, depth=1, limit=20)
print(f"Rel map for Alice-{tag} ({len(rel_map)} rows):")
for src, rel, dst in rel_map:
    print(f"  {src.name}  --[{rel.label}]-->  {dst.name}")

## 4. structured_query — arbitrary Cypher

In [ ]:
rows = store.structured_query(
    "MATCH (p:Person)-[r:RESEARCHES]->(t:Topic)"
    " WHERE p.name STARTS WITH $prefix"
    " RETURN p.name AS person, t.name AS topic, r.since AS since",
    param_map={"prefix": f"Alice-{tag}"},
)
print("Query result:", rows)

## 5. get_schema

In [ ]:
schema = store.get_schema()
print(schema)

## 6. Idempotency — double upsert must not duplicate edges

In [ ]:
store.upsert_relations(relations)  # second call — should still be exactly 1 edge
rows = store.structured_query(
    "MATCH (a {name: $src})-[r:RESEARCHES]->(b {name: $dst}) RETURN count(r) AS cnt",
    param_map={"src": f"Alice-{tag}", "dst": f"GraphRAG-{tag}"},
)
print("Edge count after double upsert (expect 1):", rows[0]["cnt"])

## Cleanup

In [ ]:
store.delete(entity_names=[f"Alice-{tag}", f"Bob-{tag}", f"GraphRAG-{tag}"])
print("Cleaned up")
store.close()
client.close()  # injected client — owned by caller